# K4TQ-G3 "Ginna 3": as fast as a 15-inch FPV quad with its payload

Ginna's airframe (Freddy × 1.35) with the Jetson Orin NX and the perception, take-off mass ≤ 12.5 kg, the payload re-sized; the drive swept for the top speed — **MN801-S KV150 + 20x14 on 12S3P**.

**The target**: the 15-inch FPV quad's top speed carrying a payload (`k4tq_quads.fpv15`, the same catalogue and drive
model: 15x8 on MN5006 KV450, 6S2P Li-ion) — ≈34.5 m/s (124 km/h); K4TQ-G3 must match or exceed it with four motors
running, with a 5 % margin (the model's 1 m/s grid and its uncertainty).

**The rules** (`k4tq_fast.sweep` / `pick`): a motor bigger than Ginna (G)'s; T/W ≥ 1.6 on the sagged pack, most loaded motor
≤ 75 % in the hover; R3 closes (10 km radius, 2 min hover, 20 % reserve, 8 m/s wind) with ≥ 20 Wh to spare; the root's
actuator (sized for the top speed + 10 %), bearings and locking pin within their ratings; the most payload that passes within 12.5 kg and Freddy's stall speed (≤ 18 m/s).

Why the drive and not the airframe: Ginna's top speed is its propeller's pitch speed, not its power —
the drag at 35 m/s is only ≈0.5 kW of thrust power. A higher pitch (or a higher pack voltage) moves it.
Labels: ESTIMATE / CALC (MOCK BEMT under `DEBUG = True`); the new parts are "to verify".

In [ ]:
DEBUG = True
from vegeta.labkit import bootstrap
DEBUG = bootstrap.lab("aero", DEBUG)

In [ ]:
import pandas as pd
from IPython.display import Image, display
import k4tq, k4tq_sizing as sz, k4tq_fast as kf, k4tq_quads as kq
pd.set_option("display.width", 240); pd.set_option("display.max_columns", 40); pd.set_option("display.max_colwidth", 50)
LETTER, PARENT = "G3", "G"
print(k4tq.params(LETTER)["title"])
print("target: the 15-inch FPV quad's top speed with payload", kf.target_speed(), "m/s")

## 1. The sweep (motor × propeller × pack) — from `data/k4tq/sweep_G3.csv`; re-run below (10–30 min)

In [ ]:
sw = pd.read_csv(sz.DATA / "sweep_G3.csv")
sw.attrs["target speed [m/s]"] = kf.target_speed()
# sw = kf.sweep(LETTER); sw.to_csv(sz.DATA / "sweep_G3.csv", index=False)       # the full sweep again
cols = ["motor", "prop", "pack", "top speed [m/s]", "take-off mass [kg]", "payload [kg]", "T/W sagged", "R3 margin [Wh]", "still-air range [km]",
        "hover [W]", "stall [m/s]", "actuator reserve", "meets the target", "note"]
sw[[c for c in cols if c in sw]].sort_values("top speed [m/s]", ascending=False).round(2)

In [ ]:
print("the pick:"); kf.pick(LETTER, sw)

## 2. Against Ginna (G) and the 15-inch FPV quad

In [ ]:
df = kf.comparison(designs=(PARENT, LETTER))
df.T

## 3. Power and range against speed

In [ ]:
Image(str(kf.figure(designs=(PARENT, LETTER))))

## 4. The root and the hover

In [ ]:
v = sz.vehicle(LETTER)
r = sz.root_loads(v)
display(pd.Series({k: r[k] for k in ("case governing the bending", "root moment ultimate [N·m]", "actuator design torque [N·m]", "actuator available at the spar [N·m]",
                    "actuator reserve", "bearing static reserve", "locking pin shear stress, double shear [MPa]", "spar chosen")}))
pd.Series(sz.hover(v))

## Reading it

- **Top speed**: ≈39 m/s (140 km/h) against the FPV's 121 km/h with 2 kg — 16 % faster with the same 2 kg payload, the
  Jetson and ≈75 km still-air range (the FPV: ≈16 km with 2 kg).
- **What it took**: the MN801-S KV150 and the 20x14 on the 12S3P: the payload falls from Ginna's 2.6 kg to 2.0 kg
  (the Jetson +0.29 kg, the motors +0.28 kg) at the same 12.5 kg; the higher pitch cruises better (≈75 km against 59 km).
- **Watch**: the tilt actuator at 43 m/s (top + 10 %) has a reserve of only ≈1.0 — a stronger servo (the 6.5 N·m class
  of `root_M`, ≈+50 g each) is the first Stage 2 change; the divergence / flutter check at 39 m/s is Stage 4.
- **The FPV still wins** the hover and the mass (4.3 kg against 12.5 kg).